In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from imblearn.over_sampling import SMOTE

import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
df = pd.read_csv('transaction.csv')
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset shape: (9841, 51)
Rows: 9841
Columns: 51


In [5]:
print(df.head())
print(df.dtypes)
print(df.info())

   Unnamed: 0  Index                                     Address  FLAG  \
0           0      1  0x00009277775ac7d0d59eaad8fee3d10ac6c805e8     0   
1           1      2  0x0002b44ddb1476db43c868bd494422ee4c136fed     0   
2           2      3  0x0002bda54cb772d040f779e88eb453cac0daa244     0   
3           3      4  0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e     0   
4           4      5  0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89     0   

   Avg min between sent tnx  Avg min between received tnx  \
0                    844.26                       1093.71   
1                  12709.07                       2958.44   
2                 246194.54                       2434.02   
3                  10219.60                      15785.09   
4                     36.61                      10707.77   

   Time Diff between first and last (Mins)  Sent tnx  Received Tnx  \
0                                704785.63       721            89   
1                               1218216.73      

In [6]:
print("Class balance:")
print(df['FLAG'].value_counts())
print()
print(f"Legitimate wallets: {df['FLAG'].value_counts()[0]}")
print(f"Fraudulent wallets: {df['FLAG'].value_counts()[1]}")
print(f"Fraud percentage: {round(df['FLAG'].value_counts()[1] / len(df) * 100, 2)}%")

Class balance:
FLAG
0    7662
1    2179
Name: count, dtype: int64

Legitimate wallets: 7662
Fraudulent wallets: 2179
Fraud percentage: 22.14%


In [9]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values: {df.isnull().any().sum()}")

Missing values per column:
Unnamed: 0                                              0
Index                                                   0
Address                                                 0
FLAG                                                    0
Avg min between sent tnx                                0
Avg min between received tnx                            0
Time Diff between first and last (Mins)                 0
Sent tnx                                                0
Received Tnx                                            0
Number of Created Contracts                             0
Unique Received From Addresses                          0
Unique Sent To Addresses                                0
min value received                                      0
max value received                                      0
avg val received                                        0
min val sent                                            0
max val sent                                 

In [8]:
# Fill missing values with 0
df = df.fillna(0)

# Confirm no missing values remain
print(f"Missing values after cleaning: {df.isnull().sum().sum()}")

Missing values after cleaning: 0


In [10]:
# Check for duplicates
print(f"Duplicate rows before cleaning: {df.duplicated().sum()}")

# Remove duplicates
df = df.drop_duplicates()

# Confirm
print(f"Duplicate rows after cleaning: {df.duplicated().sum()}")
print(f"Dataset shape after removing duplicates: {df.shape}")

Duplicate rows before cleaning: 0
Duplicate rows after cleaning: 0
Dataset shape after removing duplicates: (9841, 51)


In [11]:
# Drop irrelevant columns
df = df.drop(columns=['Address', 'Unnamed: 0', 'Index'])

# Confirm
print(f"Dataset shape after dropping irrelevant columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Dataset shape after dropping irrelevant columns: (9841, 48)
Remaining columns: ['FLAG', 'Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received ', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance', ' Total ERC20 tnxs', ' ERC20 total Ether received', ' ERC20 total ether sent', ' ERC20 total Ether sent contract', ' ERC20 uniq sent addr', ' ERC20 uniq rec addr', ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr', ' ERC20 avg time between sent tnx', ' ERC20 avg time between rec tnx', ' ERC20 avg time between rec 2 tnx', ' 

In [12]:
# Separate features (X) from target label (y)
X = df.drop(columns=['FLAG'])
y = df['FLAG']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.shape[1]}")

Features shape: (9841, 47)
Target shape: (9841,)
Feature columns: 47


In [13]:
# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Training fraud cases: {y_train.sum()}")
print(f"Testing fraud cases: {y_test.sum()}")

Training set size: (7872, 47)
Testing set size: (1969, 47)
Training fraud cases: 1743
Testing fraud cases: 436


In [14]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Training set size: {X_train.shape}")
print(f"After SMOTE - Training set size: {X_train_smote.shape}")
print(f"Before SMOTE - Fraud cases: {y_train.sum()}")
print(f"After SMOTE - Fraud cases: {y_train_smote.sum()}")
print(f"After SMOTE - Legitimate cases: {(y_train_smote == 0).sum()}")

ValueError: could not convert string to float: ' '